# 07.2 — Weighted ensemble of RT-DETR and Faster R-CNN
The best RT-DETR and the best Faster R-CNN, with test-time augmentation (full frame + flip + tiles), fused by Weighted Boxes Fusion. The setting is the best one of the reference ensemble: RT-DETR with twice the weight, fusion IoU 0.7-0.8; one run without TTA is the control.

> **Earlier version, kept for its results.** This notebook ran on Kaggle with the code of commit `046347a` (package `src/vod`, later replaced by `src/core`). Validation here is the official val split, not the folds. It reuses the outputs of the earlier 07 notebooks. To rerun it, clone that commit instead of `main`.

In [1]:
import subprocess
import sys
from pathlib import Path

if Path("../src/vod").exists():
    sys.path.insert(0, "../src")
else:
    repo = Path("/tmp/repo")  # outside /kaggle/working so it is not saved as output
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/mohinska/ai_hw2_MultipleObjectVideoDetection", str(repo)], check=True)
    sys.path.insert(0, str(repo / "src"))

Cloning into '/tmp/repo'...


In [2]:
!pip install -q ultralytics faster-coco-eval

In [3]:
import re

import pandas as pd
import torch

from vod import CLASSES
from vod.data import find_data_root, load_split
from vod.evaluate import coco_evaluate, per_clip_map
from vod.protocol import PROTOCOL
from vod.submission import write_submission
from vod.tv import comparison_tables, completed_runs, load_run_model
from vod.variants import (choose_variant, collect_view, compare_variants, merge_views, torchvision_detector,
                          yolo_detector)

DATA = find_data_root()
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../outputs")
WORK.mkdir(parents=True, exist_ok=True)
ROOTS = (WORK, Path("/kaggle/input"))

## Models

In [4]:
def has_weights(pattern):
    return any(root.exists() and any(root.rglob(pattern)) for root in ROOTS)


def make_detector(name):
    if name.startswith(("yolo", "rtdetr")):
        from vod.yolo import load_ultralytics
        pattern = "baseline/weights/best.pt" if name == "yolo_baseline" else f"{name}/weights/last.pt"
        weights = next(p for root in ROOTS if root.exists() for p in root.rglob(pattern))
        return yolo_detector(load_ultralytics(weights))
    cfg = runs[name]["config"]
    return torchvision_detector(load_run_model(name, cfg, ROOTS), cfg)


runs = completed_runs(ROOTS)
candidates = ["yolo_baseline"] if has_weights("baseline/weights/best.pt") else []
if runs:
    main = comparison_tables(runs, CLASSES)[0]
    for prefix, pattern in [("yolo_", "{}/weights/last.pt"), ("rtdetr", "{}/weights/last.pt"),
                            ("frcnn", "{}/last.pt"), ("retina", "{}/last.pt"), ("fcos", "{}/last.pt")]:
        family = [n for n in main[main.index.str.startswith(prefix)].sort_values("mAP", ascending=False).index
                  if has_weights(pattern.format(n))]
        candidates += family[:1]
print("candidates:", candidates)
assert candidates, "no trained weights found: add the outputs of the training notebooks (runs/<name>/weights/last.pt, metrics.json) as Input"

candidates: ['rtdetr_lr3e4', 'frcnn_default']


In [5]:
MEMBERS = [next((n for n in candidates if n.startswith(f)), None) for f in ("rtdetr", "frcnn")]
assert all(MEMBERS), f"need both an RT-DETR and a Faster R-CNN run with weights, found {MEMBERS}"
print("ensemble:", MEMBERS)

VAL_STEP, LOW_RES, BATCH = 5, 960, 4
TTA = ["full", "flip", "tiles"]
TTA_IOU = 0.65  # NMS between the views of one model, as the best variant of 07
val_frames, _ = load_split(DATA / "val", step=1)
VAL_GT = DATA / "val" / "annotations_coco.json"
paths = val_frames.iloc[::VAL_STEP]["path"].tolist()
print(len(paths), "val frames")


def collect_all(frames, max_side=None):
    """Views {(model, view): detections} of both models."""
    views = {}
    for name in MEMBERS:
        detect = make_detector(name)
        for v in TTA:
            views[name, v] = collect_view(detect, frames, v, max_side=max_side, batch=BATCH)
        del detect
        torch.cuda.empty_cache()
    return views


def build(views, keys, tta, weights=None, iou=None):
    """Detections of the models `keys`, each with or without TTA, fused by WBF (a single model is not fused)."""
    parts = [merge_views([views[k, v] for v in (TTA if tta else ["full"])], TTA_IOU if tta else None) for k in keys]
    return parts[0] if len(parts) == 1 else merge_views(parts, iou, fuse="wbf", weights=weights)


def score(frames, dets):
    m = coco_evaluate(VAL_GT, dets, file_names=frames)
    return {k: round(m[k], 4) for k in ["mAP", "AP50", "AP_small", "AR500"]}

ensemble: ['rtdetr_lr3e4', 'frcnn_default']
570 val frames


## Sweep on val (native resolution)

In [6]:
views = collect_all(paths)
configs = {}
for name in MEMBERS:
    for tta in (False, True):
        configs[f"{name}{' TTA' if tta else ''} (single model)"] = ([name], tta, None, None)
# the best setting of the reference ensemble: TTA, RT-DETR twice the weight, high fusion IoU; the run without TTA is the control
configs[f"{MEMBERS[0]} + {MEMBERS[1]} 2:1 iou 0.7 (no TTA)"] = (MEMBERS, False, (2, 1), 0.7)
for w in ((2, 1), (3, 1)):
    for iou in (0.7, 0.8):
        configs[f"{MEMBERS[0]} TTA + {MEMBERS[1]} TTA {w[0]}:{w[1]} iou {iou}"] = (MEMBERS, True, w, iou)
table = pd.DataFrame({name: score(paths, build(views, *cfg)) for name, cfg in configs.items()}).T
table.to_csv(WORK / "ensemble_summary.csv")
table

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


,mAP,AP50,AP_small,AR500
rtdetr_lr3e4 (single model),0.2641,0.4978,0.1178,0.4074
rtdetr_lr3e4 TTA (single model),0.2870,0.5404,0.1769,0.4514
frcnn_default (single model),0.2101,0.4262,0.1124,0.4149
frcnn_default TTA (single model),0.2281,0.4697,0.1613,0.4527
rtdetr_lr3e4 + frcnn_default 2:1 iou 0.7 (no TTA),0.2661,0.5127,0.1275,0.4865
rtdetr_lr3e4 TTA + frcnn_default TTA 2:1 iou 0.7,0.2892,0.5502,0.1932,0.4634
rtdetr_lr3e4 TTA + frcnn_default TTA 2:1 iou 0.8,0.2845,0.5359,0.1696,0.4676
rtdetr_lr3e4 TTA + frcnn_default TTA 3:1 iou 0.7,0.2921,0.5532,0.1898,0.4629
rtdetr_lr3e4 TTA + frcnn_default TTA 3:1 iou 0.8,0.2882,0.5419,0.1703,0.4674


## Best ensemble at 960 px
Shrunk frames imitate the low-res test clips.

In [7]:
best = table.loc[[n for n in table.index if "single" not in n], "mAP"].idxmax()
best_single = table.loc[[n for n in table.index if "single" in n], "mAP"].idxmax()
print("best ensemble:", best, "| best single:", best_single)
views_low = collect_all(paths, LOW_RES)
low = pd.DataFrame({n: score(paths, build(views_low, *configs[n])) for n in (best_single, best)}).T
low["mAP_native"] = table.loc[[best_single, best], "mAP"]
clips = {n: pd.Series(per_clip_map(VAL_GT, build(views, *configs[n]), paths)) for n in (best_single, best)}
clips_better = int((clips[best] > clips[best_single]).sum())
print("clips better than the best single model:", clips_better, "of", len(clips[best]))
low

best ensemble: rtdetr_lr3e4 TTA + frcnn_default TTA 3:1 iou 0.7 | best single: rtdetr_lr3e4 TTA (single model)
clips better than the best single model: 7 of 7


,mAP,AP50,AP_small,AR500,mAP_native
rtdetr_lr3e4 TTA (single model),0.2649,0.5027,0.1371,0.4324,0.2870
rtdetr_lr3e4 TTA + frcnn_default TTA 3:1 iou 0.7,0.2754,0.5222,0.1602,0.4456,0.2921


## Test predictions
The ensemble is submitted only if it beats the best single model at both resolutions.

In [8]:
use_ensemble = (table.loc[best, "mAP"] > table.loc[best_single, "mAP"] and low.loc[best, "mAP"] > low.loc[best_single, "mAP"]
                and clips_better > len(clips[best]) / 2)
final = best if use_ensemble else best_single
print("submitting:", final)
test = pd.read_csv(DATA / "test_images.csv")
test_paths = [str(DATA / f) for f in test["file_name"]]
dets = build(collect_all(test_paths), *configs[final])
dets.to_parquet(WORK / "dets_test.parquet")  # ~1.5 h of inference, kept if a later cell fails
dets["image_id"] = dets.pop("file_name").map(dict(zip(test_paths, test["image_id"])))
tag = re.sub(r"[^\w.+-]", "_", final)  # no spaces or colons in a file name
sub = write_submission(dets, DATA / "test_images.csv", WORK / f"submission_{tag}.csv")
sub.head()

submitting: rtdetr_lr3e4 TTA + frcnn_default TTA 3:1 iou 0.7


,image_id,PredictionString
0,clip_001_0000001,0 0.8659 631.4 916.6 63.8 159.6 0 0.8641 782.2...
1,clip_001_0000002,0 0.8678 626.3 924.5 63.7 152.7 0 0.8517 781.3...
2,clip_001_0000003,0 0.8678 773.2 897.6 63.6 176.0 0 0.8669 617.2...
3,clip_001_0000004,0 0.8534 762.0 902.7 63.4 173.3 0 0.8465 1669....
4,clip_001_0000005,0 0.8732 560.5 967.5 68.6 111.1 0 0.8709 731.0...
